In [1]:
import scipy
import sys
from scipy import io
import numpy as np
import numpy.matlib
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
import pandas as pd
from numpy import matlib
import seaborn as sns
import math
import os
import scipy.optimize
import scipy.io as sio
from scipy import stats
import scipy.ndimage

In [2]:
##################### Experiment specifics
cutoff = None

expt_labels = ['bFB69_Tun_gr', 'bFB87_Tun_gr', 'bFB93_Tun_gr']
pert=r'0.5 $\mu$g/mL Tunicamycin added'

linestyles = ['-.', '-', '--', '.']
out_path = '/Users/barber.527/Documents/GitHub/Rojas_lab_drafts/outputs/compiled_data/growth_rate_plots/'
in_path = '/Users/barber.527/Documents/GitHub/Rojas_lab_drafts/outputs/compiled_data/'

plot_labels = []

# loading the experimental parameters
for expt_label in expt_labels:
    temp_path = in_path + expt_label + '_condition_parameters.pkl'
    with open(temp_path, 'rb') as input:
        expt_vals = pickle.load(input)
    plot_labels.append(expt_vals['Celltype'])
vars = expt_vals['vars']  # This part should be conserved across all compiled experiments
normed = ['_normed', '']
# ylabels = {'sgr_l': r'$\frac{1}{L}\frac{dL}{dt}$', 'sgr_sa': r'$\frac{1}{S}\frac{dS}{dt}$', 'width':'Width'}
ylabels = {'sgr_l': r'Growth rate $\lambda_l$', 'sgr_sa': r'Growth rate $\lambda_S$', 'width': 'Width'}
yunits = {'sgr_l': r' $(h^{-1})$', 'sgr_sa': r' ($h^{-1}$)', 'width': r' ($\mu$m)'}
sel_vars = ['sgr_l', 'width']

Now we import our data and test significance across all variables listed

In [4]:
norm=''
time_testers=[-5,45] # time in minutes for test to take place
for time_tester in time_testers:
    print('Time =', time_tester)
    for var in sel_vars:
        xvs=[]
        yvs=[]
        fig = plt.figure(figsize=[10, 6])
        for ind in range(len(expt_labels)):
            expt=expt_labels[ind]
            xv = np.load(in_path + expt + '_time.npy')
            if not cutoff is None:
                temp_vals = np.nonzero(xv / 60 < cutoff)[0][-1]
            else:
                temp_vals = len(xv) - 1
            xvs.append(xv[:temp_vals])
            yvs.append(np.load(in_path + expt + '_' + var + norm + '.npy')[:, :temp_vals])
        
        ind_vals=[np.nonzero(xvs[ind]==time_tester*60.0)[0][0] for ind in range(len(xvs))]
        for ind in range(len(ind_vals)):
            for ind1 in np.arange(ind+1,len(ind_vals)):
                print('Comparing conditions: ', expt_labels[ind],expt_labels[ind1])
                print('Variable = ',var)
                # print(xvs[ind][ind_vals[ind]]/60.0,xvs[ind1][ind_vals[ind1]]/60.0)
                temp_yv1=yvs[ind][:,ind_vals[ind]]
                temp_yv1=temp_yv1[np.nonzero(~np.isnan(temp_yv1))]
                temp_yv2=yvs[ind1][:,ind_vals[ind1]]
                temp_yv2=temp_yv2[np.nonzero(~np.isnan(temp_yv2))]
                print(scipy.stats.ttest_ind(temp_yv1,temp_yv2))

Time = -5
Comparing conditions:  bFB69_Tun_gr bFB87_Tun_gr
Variable =  sgr_l
Ttest_indResult(statistic=5.669445367202726, pvalue=3.2801998789349095e-08)
Comparing conditions:  bFB69_Tun_gr bFB93_Tun_gr
Variable =  sgr_l
Ttest_indResult(statistic=4.964129121780575, pvalue=1.1897274192838599e-06)
Comparing conditions:  bFB87_Tun_gr bFB93_Tun_gr
Variable =  sgr_l
Ttest_indResult(statistic=0.5550951475828056, pvalue=0.5805265958283832)
Comparing conditions:  bFB69_Tun_gr bFB87_Tun_gr
Variable =  width
Ttest_indResult(statistic=-3.677406774280483, pvalue=0.000261848044363437)
Comparing conditions:  bFB69_Tun_gr bFB93_Tun_gr
Variable =  width
Ttest_indResult(statistic=-5.857286605003807, pvalue=8.784413653127002e-09)
Comparing conditions:  bFB87_Tun_gr bFB93_Tun_gr
Variable =  width
Ttest_indResult(statistic=-1.8210494078051553, pvalue=0.07041079200183963)
Time = 45
Comparing conditions:  bFB69_Tun_gr bFB87_Tun_gr
Variable =  sgr_l
Ttest_indResult(statistic=4.098000940609967, pvalue=6.012001